Run this %sh cell once in your notebook before the runner script. This deletes any **ghost folders** that are blocking the process.

In [0]:
%sh
find /Workspace/Repos/prateek.k.jha@v4c.ai/vstone-databricks-pipeline/tests/ -name "__pycache__" -type d -exec rm -rf {} +
find /Workspace/Repos/prateek.k.jha@v4c.ai/vstone-databricks-pipeline/tests/ -name ".pytest_cache" -type d -exec rm -rf {} +

Running the main QA code for Bronze layer from here and it is to be added in bronze layer ingestion pipeline job as one of the last steps

In [0]:
import pytest
import os
import sys
import shutil

# --- STEP 1: AGGRESSIVE CACHE SUPPRESSION ---
# Prevent Python from ever trying to write .pyc files in the Repo
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"

# --- STEP 2: PATH SETUP ---
target_path = "/Workspace/Repos/prateek.k.jha@v4c.ai/vstone-databricks-pipeline/tests/"

if not os.path.exists(target_path):
    raise FileNotFoundError(f"Directory not found: {target_path}")

os.chdir(target_path)

if target_path not in sys.path:
    sys.path.insert(0, target_path)

# --- STEP 3: RUN PYTEST WITH REDIRECTED CACHE ---
print(f"Starting Bronze Layer QA Tests in {target_path}...\n")

# Logic:
# -o cache_dir=/tmp/... : Moves the metadata cache to the cluster's local SSD
# -p no:cacheprovider   : Disables the cache plugin entirely to be safe
# --import-mode=importlib: Use the modern importer to avoid path collisions
exit_code = pytest.main([
    "-v", 
    "-s", 
    "test_bronze_layer.py",
    "-p", "no:cacheprovider",
    "-o", "cache_dir=/tmp/prateek_pytest_cache", 
    "--import-mode=importlib",
    "--color=yes"
])

# --- STEP 4: EXPLICIT LOGGING & JOB FAILURE ---
if exit_code == 0:
    print("\n" + "="*40)
    print("SUCCESS: All Bronze Layer Tests Passed!")
    print("="*40)
else:
    # Map common exit codes for better logging
    error_map = {1: "Tests Failed", 2: "Interrupted", 3: "Internal Error", 4: "Usage/Collection Error", 5: "No tests found"}
    msg = error_map.get(exit_code, "Unknown Error")
    
    print("\n" + "!"*40)
    print(f"FAILURE: {msg} (Exit Code {exit_code})")
    print("Check the verbose output above for specific assertion errors.")
    print("!"*40)
    
    raise Exception(f"Quality Gate Failed. Pytest Exit Code: {exit_code} ({msg})")